Fix missing database context for temp table creation in population cleaning cell
*Co-authored with CoCo*

# Supplementary Data Cleaning and Validation in Snowflake

This notebook cleans the three enrichment sources used by the police-crime project:

1. ONS police-force-area population estimates, 2021-2024.
2. England Index of Multiple Deprivation (IMD) 2025.
3. Wales Index of Multiple Deprivation (WIMD) 2025.

It reads external Snowflake stages without modifying them and publishes separate clean,
quarantine, audit, and validation tables. England IMD and Wales WIMD remain explicitly
identified because their scores and ranks are not directly comparable.

The notebook does not perform the enrichment join. It prepares validated handoff files
for the enrichment owner.

## Expected source grain

| Dataset | Raw structure | Clean grain |
|---|---|---|
| Population | One force/year row with 172 age-sex columns | `FORCE_NAME + YEAR` |
| England IMD | One row per 2021 LSOA | `LSOA_CODE` |
| Wales WIMD | Long format: LSOA × domain × measure | `LSOA_CODE` after pivot |

The England crime-domain measures and Wales community-safety measures are preserved
but clearly flagged as unsuitable for explaining police crime because they can create
circular analysis.

In [ ]:
from datetime import datetime, timezone
import re
import uuid

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception as exc:
    raise RuntimeError(
        "Run this notebook inside Snowflake or provide a configured Snowpark Session."
    ) from exc

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

# Raw external stages
POPULATION_STAGE = "CRIME_ETL_DB.RAW.POPULATION_S3_STAGE"
ENGLAND_STAGE = "CRIME_ETL_DB.RAW.DEPRIVATION_ENGLAND_S3_STAGE"
WALES_STAGE = "CRIME_ETL_DB.RAW.DEPRIVATION_WALES_S3_STAGE"

# Published Snowflake tables
CLEAN_SCHEMA = "CRIME_ETL_DB.DATA_QUALITY"
POPULATION_CLEAN = f"{CLEAN_SCHEMA}.POPULATION_CLEAN"
POPULATION_QUARANTINE = f"{CLEAN_SCHEMA}.POPULATION_QUARANTINE"
ENGLAND_CLEAN = f"{CLEAN_SCHEMA}.DEPRIVATION_ENGLAND_CLEAN"
ENGLAND_QUARANTINE = f"{CLEAN_SCHEMA}.DEPRIVATION_ENGLAND_QUARANTINE"
WALES_LONG_CLEAN = f"{CLEAN_SCHEMA}.DEPRIVATION_WALES_LONG_CLEAN"
WALES_CLEAN = f"{CLEAN_SCHEMA}.DEPRIVATION_WALES_CLEAN"
WALES_QUARANTINE = f"{CLEAN_SCHEMA}.DEPRIVATION_WALES_QUARANTINE"
AUDIT_TABLE = f"{CLEAN_SCHEMA}.SUPPLEMENTARY_CLEANING_AUDIT"
VALIDATION_TABLE = f"{CLEAN_SCHEMA}.SUPPLEMENTARY_VALIDATION_RESULTS"

# Existing external clean stage shared with the police-data exports.
# Set EXPORT_TO_S3=False only when this stage is temporarily unavailable.
EXPORT_TO_S3 = True
CLEAN_EXPORT_STAGE = "CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE"

IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_$]*(\.[A-Za-z_][A-Za-z0-9_$]*){0,2}$")
for name in [
    POPULATION_STAGE, ENGLAND_STAGE, WALES_STAGE, CLEAN_SCHEMA,
    POPULATION_CLEAN, POPULATION_QUARANTINE, ENGLAND_CLEAN,
    ENGLAND_QUARANTINE, WALES_LONG_CLEAN, WALES_CLEAN,
    WALES_QUARANTINE, AUDIT_TABLE, VALIDATION_TABLE,
    CLEAN_EXPORT_STAGE,
]:
    if not IDENTIFIER.fullmatch(name):
        raise ValueError(f"Unsafe Snowflake identifier: {name}")

print(f"Supplementary cleaning run: {RUN_ID}")

## 1. Confirm source stages and create audit tables

In [ ]:
session.sql(f"CREATE SCHEMA IF NOT EXISTS {CLEAN_SCHEMA}").collect()

for stage in [POPULATION_STAGE, ENGLAND_STAGE, WALES_STAGE]:
    files = session.sql(f"LIST @{stage}").collect()
    if not files:
        raise ValueError(f"No files found in @{stage}")
    print(f"PASS - @{stage}: {len(files)} file(s)")

session.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
    RUN_ID STRING,
    DATASET STRING,
    STARTED_AT TIMESTAMP_TZ,
    COMPLETED_AT TIMESTAMP_TZ,
    RAW_ROWS NUMBER,
    CLEAN_ROWS NUMBER,
    QUARANTINE_ROWS NUMBER,
    STATUS STRING
)
""").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {VALIDATION_TABLE} (
    RUN_ID STRING,
    DATASET STRING,
    CHECK_NAME STRING,
    SEVERITY STRING,
    OBSERVED_VALUE STRING,
    EXPECTED_VALUE STRING,
    PASSED BOOLEAN,
    CHECKED_AT TIMESTAMP_TZ
)
""").collect()

## 2. Population cleaning

The ONS source contains 172 population columns: female ages `F0-F85` and male ages
`M0-M85`. The final population is their sum. Only observed years 2021-2024 are
published; 2025 and 2026 are not estimated in this cleaning layer.

In [ ]:
# Positional CSV columns: $1 code, $2 force, $3 year, $4-$175 age/sex values.
session.sql("USE DATABASE CRIME_ETL_DB").collect()
session.sql("USE SCHEMA RAW").collect()

age_positions = range(4, 176)
population_sum_sql = " + ".join(
    f"COALESCE(TRY_TO_NUMBER(t.${position}), 0)" for position in age_positions
)
invalid_age_sql = " + ".join(
    "IFF(NULLIF(TRIM(t.${0}), '') IS NOT NULL "
    "AND TRY_TO_NUMBER(t.${0}) IS NULL, 1, 0)".format(position)
    for position in age_positions
)

session.sql(f"""
CREATE OR REPLACE TEMPORARY TABLE STG_POPULATION AS
WITH PARSED AS (
    SELECT
        UPPER(NULLIF(TRIM(t.$1), '')) AS PFA_CODE,
        NULLIF(TRIM(t.$2), '') AS FORCE_NAME_RAW,
        TRY_TO_NUMBER(t.$3) AS YEAR,
        ({population_sum_sql}) AS TOTAL_POPULATION,
        ({invalid_age_sql}) AS INVALID_AGE_CELLS,
        METADATA$FILENAME AS SOURCE_FILE_NAME
    FROM @{POPULATION_STAGE} t
    WHERE TRY_TO_NUMBER(t.$3) BETWEEN 2021 AND 2024
), STANDARDISED AS (
    SELECT *,
        CASE LOWER(FORCE_NAME_RAW)
            WHEN 'metropolitan police' THEN 'Metropolitan Police Service'
            WHEN 'metropolitan police service' THEN 'Metropolitan Police Service'
            WHEN 'west midlands' THEN 'West Midlands Police'
            WHEN 'west midlands police' THEN 'West Midlands Police'
            WHEN 'south wales' THEN 'South Wales Police'
            WHEN 'south wales police' THEN 'South Wales Police'
            WHEN 'sussex' THEN 'Sussex Police'
            WHEN 'sussex police' THEN 'Sussex Police'
        END AS FORCE_NAME
    FROM PARSED
), ASSESSED AS (
    SELECT *,
        ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
            IFF(PFA_CODE IS NULL, 'MISSING_PFA_CODE', NULL),
            IFF(FORCE_NAME IS NULL, 'OUT_OF_SCOPE_OR_UNKNOWN_FORCE', NULL),
            IFF(YEAR NOT BETWEEN 2021 AND 2024, 'INVALID_YEAR', NULL),
            IFF(INVALID_AGE_CELLS > 0, 'NON_NUMERIC_AGE_VALUE', NULL),
            IFF(TOTAL_POPULATION <= 0, 'INVALID_TOTAL_POPULATION', NULL)
        ), '|') AS BASE_REJECTION_REASON,
        ROW_NUMBER() OVER (
            PARTITION BY FORCE_NAME, YEAR
            ORDER BY SOURCE_FILE_NAME
        ) AS GRAIN_RANK
    FROM STANDARDISED
    WHERE FORCE_NAME IS NOT NULL
)
SELECT *,
    CASE
        WHEN BASE_REJECTION_REASON <> '' AND GRAIN_RANK > 1
            THEN BASE_REJECTION_REASON || '|DUPLICATE_FORCE_YEAR'
        WHEN BASE_REJECTION_REASON <> '' THEN BASE_REJECTION_REASON
        WHEN GRAIN_RANK > 1 THEN 'DUPLICATE_FORCE_YEAR'
    END AS REJECTION_REASON
FROM ASSESSED
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {POPULATION_CLEAN} AS
SELECT
    PFA_CODE, FORCE_NAME, YEAR::INTEGER AS YEAR,
    TOTAL_POPULATION::INTEGER AS TOTAL_POPULATION,
    FALSE AS IS_ESTIMATED,
    SOURCE_FILE_NAME,
    '{RUN_ID}' AS CLEANING_RUN_ID,
    CURRENT_TIMESTAMP() AS CLEANED_AT
FROM STG_POPULATION
WHERE REJECTION_REASON IS NULL
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {POPULATION_QUARANTINE} AS
SELECT *, '{RUN_ID}' AS CLEANING_RUN_ID, CURRENT_TIMESTAMP() AS QUARANTINED_AT
FROM STG_POPULATION
WHERE REJECTION_REASON IS NOT NULL
""").collect()

## 3. England IMD cleaning

The clean handoff retains the overall index and non-crime socioeconomic domains.
Crime-domain fields are preserved with an exclusion flag but should not be used as
independent explanatory variables for police crime.

In [ ]:
session.sql(f"""
CREATE OR REPLACE TEMPORARY TABLE STG_ENGLAND_IMD AS
WITH PARSED AS (
    SELECT
        UPPER(NULLIF(TRIM(t.$1), '')) AS LSOA_CODE,
        NULLIF(TRIM(t.$2), '') AS LSOA_NAME,
        UPPER(NULLIF(TRIM(t.$3), '')) AS LOCAL_AUTHORITY_CODE,
        NULLIF(TRIM(t.$4), '') AS LOCAL_AUTHORITY_NAME,
        TRY_TO_DECIMAL(t.$5, 10, 3) AS IMD_SCORE,
        TRY_TO_NUMBER(t.$6) AS IMD_RANK,
        TRY_TO_NUMBER(t.$7) AS IMD_DECILE,
        TRY_TO_DECIMAL(t.$8, 10, 3) AS INCOME_SCORE,
        TRY_TO_NUMBER(t.$9) AS INCOME_RANK,
        TRY_TO_NUMBER(t.$10) AS INCOME_DECILE,
        TRY_TO_DECIMAL(t.$11, 10, 3) AS EMPLOYMENT_SCORE,
        TRY_TO_NUMBER(t.$12) AS EMPLOYMENT_RANK,
        TRY_TO_NUMBER(t.$13) AS EMPLOYMENT_DECILE,
        TRY_TO_DECIMAL(t.$14, 10, 3) AS EDUCATION_SCORE,
        TRY_TO_NUMBER(t.$15) AS EDUCATION_RANK,
        TRY_TO_NUMBER(t.$16) AS EDUCATION_DECILE,
        TRY_TO_DECIMAL(t.$17, 10, 3) AS HEALTH_SCORE,
        TRY_TO_NUMBER(t.$18) AS HEALTH_RANK,
        TRY_TO_NUMBER(t.$19) AS HEALTH_DECILE,
        TRY_TO_DECIMAL(t.$20, 10, 3) AS CRIME_SCORE,
        TRY_TO_NUMBER(t.$21) AS CRIME_RANK,
        TRY_TO_NUMBER(t.$22) AS CRIME_DECILE,
        TRY_TO_DECIMAL(t.$23, 10, 3) AS HOUSING_BARRIERS_SCORE,
        TRY_TO_NUMBER(t.$24) AS HOUSING_BARRIERS_RANK,
        TRY_TO_NUMBER(t.$25) AS HOUSING_BARRIERS_DECILE,
        TRY_TO_DECIMAL(t.$26, 10, 3) AS LIVING_ENVIRONMENT_SCORE,
        TRY_TO_NUMBER(t.$27) AS LIVING_ENVIRONMENT_RANK,
        TRY_TO_NUMBER(t.$28) AS LIVING_ENVIRONMENT_DECILE,
        TRY_TO_NUMBER(t.$53) AS TOTAL_POPULATION_MID_2022,
        TRY_TO_NUMBER(t.$54) AS DEPENDENT_CHILDREN_0_15_MID_2022,
        TRY_TO_NUMBER(t.$55) AS OLDER_POPULATION_60_PLUS_MID_2022,
        TRY_TO_NUMBER(t.$56) AS WORKING_AGE_POPULATION_MID_2022,
        METADATA$FILENAME AS SOURCE_FILE_NAME
    FROM @{ENGLAND_STAGE} t
    WHERE NULLIF(TRIM(t.$1), '') IS NOT NULL
      AND LOWER(TRIM(t.$1)) <> 'lsoa code (2021)'
), ASSESSED AS (
    SELECT *,
        ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
            IFF(NOT REGEXP_LIKE(LSOA_CODE, '^E01[0-9]{{6}}$'), 'INVALID_LSOA_CODE', NULL),
            IFF(LSOA_NAME IS NULL, 'MISSING_LSOA_NAME', NULL),
            IFF(IMD_SCORE IS NULL, 'INVALID_IMD_SCORE', NULL),
            IFF(IMD_RANK IS NULL OR IMD_RANK < 1, 'INVALID_IMD_RANK', NULL),
            IFF(IMD_DECILE NOT BETWEEN 1 AND 10, 'INVALID_IMD_DECILE', NULL),
            IFF(INCOME_DECILE NOT BETWEEN 1 AND 10, 'INVALID_INCOME_DECILE', NULL),
            IFF(EMPLOYMENT_DECILE NOT BETWEEN 1 AND 10, 'INVALID_EMPLOYMENT_DECILE', NULL),
            IFF(EDUCATION_DECILE NOT BETWEEN 1 AND 10, 'INVALID_EDUCATION_DECILE', NULL),
            IFF(HEALTH_DECILE NOT BETWEEN 1 AND 10, 'INVALID_HEALTH_DECILE', NULL),
            IFF(HOUSING_BARRIERS_DECILE NOT BETWEEN 1 AND 10,
                'INVALID_HOUSING_BARRIERS_DECILE', NULL),
            IFF(LIVING_ENVIRONMENT_DECILE NOT BETWEEN 1 AND 10,
                'INVALID_LIVING_ENVIRONMENT_DECILE', NULL),
            IFF(TOTAL_POPULATION_MID_2022 IS NULL OR TOTAL_POPULATION_MID_2022 < 0,
                'INVALID_POPULATION', NULL)
        ), '|') AS BASE_REJECTION_REASON,
        ROW_NUMBER() OVER (PARTITION BY LSOA_CODE ORDER BY SOURCE_FILE_NAME) AS LSOA_RANK
    FROM PARSED
)
SELECT *,
    CASE
        WHEN BASE_REJECTION_REASON <> '' AND LSOA_RANK > 1
            THEN BASE_REJECTION_REASON || '|DUPLICATE_LSOA'
        WHEN BASE_REJECTION_REASON <> '' THEN BASE_REJECTION_REASON
        WHEN LSOA_RANK > 1 THEN 'DUPLICATE_LSOA'
    END AS REJECTION_REASON
FROM ASSESSED
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {ENGLAND_CLEAN} AS
SELECT
    LSOA_CODE, LSOA_NAME, LOCAL_AUTHORITY_CODE, LOCAL_AUTHORITY_NAME,
    IMD_SCORE, IMD_RANK::INTEGER AS IMD_RANK, IMD_DECILE::INTEGER AS IMD_DECILE,
    INCOME_SCORE, INCOME_RANK::INTEGER AS INCOME_RANK,
    INCOME_DECILE::INTEGER AS INCOME_DECILE,
    EMPLOYMENT_SCORE, EMPLOYMENT_RANK::INTEGER AS EMPLOYMENT_RANK,
    EMPLOYMENT_DECILE::INTEGER AS EMPLOYMENT_DECILE,
    EDUCATION_SCORE, EDUCATION_RANK::INTEGER AS EDUCATION_RANK,
    EDUCATION_DECILE::INTEGER AS EDUCATION_DECILE,
    HEALTH_SCORE, HEALTH_RANK::INTEGER AS HEALTH_RANK,
    HEALTH_DECILE::INTEGER AS HEALTH_DECILE,
    CRIME_SCORE, CRIME_RANK::INTEGER AS CRIME_RANK,
    CRIME_DECILE::INTEGER AS CRIME_DECILE,
    TRUE AS EXCLUDE_CRIME_DOMAIN_FROM_CRIME_ANALYSIS,
    HOUSING_BARRIERS_SCORE,
    HOUSING_BARRIERS_RANK::INTEGER AS HOUSING_BARRIERS_RANK,
    HOUSING_BARRIERS_DECILE::INTEGER AS HOUSING_BARRIERS_DECILE,
    LIVING_ENVIRONMENT_SCORE,
    LIVING_ENVIRONMENT_RANK::INTEGER AS LIVING_ENVIRONMENT_RANK,
    LIVING_ENVIRONMENT_DECILE::INTEGER AS LIVING_ENVIRONMENT_DECILE,
    TOTAL_POPULATION_MID_2022::INTEGER AS TOTAL_POPULATION_MID_2022,
    DEPENDENT_CHILDREN_0_15_MID_2022::INTEGER AS DEPENDENT_CHILDREN_0_15_MID_2022,
    OLDER_POPULATION_60_PLUS_MID_2022::INTEGER AS OLDER_POPULATION_60_PLUS_MID_2022,
    WORKING_AGE_POPULATION_MID_2022::INTEGER AS WORKING_AGE_POPULATION_MID_2022,
    'IMD' AS INDEX_NAME, 2025 AS INDEX_YEAR, 'England' AS NATION,
    'LSOA 2021' AS GEOGRAPHY_VERSION,
    SOURCE_FILE_NAME, '{RUN_ID}' AS CLEANING_RUN_ID,
    CURRENT_TIMESTAMP() AS CLEANED_AT
FROM STG_ENGLAND_IMD
WHERE REJECTION_REASON IS NULL
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {ENGLAND_QUARANTINE} AS
SELECT *, '{RUN_ID}' AS CLEANING_RUN_ID, CURRENT_TIMESTAMP() AS QUARANTINED_AT
FROM STG_ENGLAND_IMD
WHERE REJECTION_REASON IS NOT NULL
""").collect()

## 4. Wales WIMD cleaning and pivot

The Wales source contains 86,265 expected rows: 1,917 LSOAs × 9 domains × 5
measures. The long clean table preserves all source measures. The pivoted table
contains ranks and deciles for enrichment.

In [ ]:
session.sql(f"""
CREATE OR REPLACE TEMPORARY TABLE STG_WALES_WIMD AS
WITH PARSED AS (
    SELECT
        TRY_TO_NUMBER(t.$1) AS MEASURE_VALUE,
        INITCAP(NULLIF(TRIM(t.$3), '')) AS MEASURE,
        UPPER(NULLIF(TRIM(t.$7), '')) AS LSOA_CODE,
        NULLIF(TRIM(t.$11), '') AS LSOA_NAME,
        NULLIF(TRIM(t.$15), '') AS DOMAIN,
        NULLIF(TRIM(t.$19), '') AS NOTES,
        METADATA$FILENAME AS SOURCE_FILE_NAME
    FROM @{WALES_STAGE} t
    WHERE NULLIF(TRIM(t.$7), '') IS NOT NULL
      AND LOWER(TRIM(t.$7)) <> 'area code'
), ASSESSED AS (
    SELECT *,
        ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
            IFF(NOT REGEXP_LIKE(LSOA_CODE, '^W01[0-9]{{6}}$'), 'INVALID_LSOA_CODE', NULL),
            IFF(LSOA_NAME IS NULL, 'MISSING_LSOA_NAME', NULL),
            IFF(DOMAIN IS NULL, 'MISSING_DOMAIN', NULL),
            IFF(MEASURE NOT IN ('Rank','Decile','Group','Quartile','Quintile'),
                'UNKNOWN_MEASURE', NULL),
            IFF(MEASURE_VALUE IS NULL, 'NON_NUMERIC_VALUE', NULL),
            IFF(MEASURE = 'Rank' AND MEASURE_VALUE NOT BETWEEN 1 AND 1917,
                'RANK_OUT_OF_RANGE', NULL),
            IFF(MEASURE = 'Decile' AND MEASURE_VALUE NOT BETWEEN 1 AND 10,
                'DECILE_OUT_OF_RANGE', NULL),
            IFF(MEASURE = 'Group' AND MEASURE_VALUE NOT BETWEEN 1 AND 5,
                'GROUP_OUT_OF_RANGE', NULL),
            IFF(MEASURE = 'Quartile' AND MEASURE_VALUE NOT BETWEEN 1 AND 4,
                'QUARTILE_OUT_OF_RANGE', NULL),
            IFF(MEASURE = 'Quintile' AND MEASURE_VALUE NOT BETWEEN 1 AND 5,
                'QUINTILE_OUT_OF_RANGE', NULL)
        ), '|') AS BASE_REJECTION_REASON,
        ROW_NUMBER() OVER (
            PARTITION BY LSOA_CODE, DOMAIN, MEASURE
            ORDER BY SOURCE_FILE_NAME
        ) AS MEASURE_RANK
    FROM PARSED
)
SELECT *,
    CASE
        WHEN BASE_REJECTION_REASON <> '' AND MEASURE_RANK > 1
            THEN BASE_REJECTION_REASON || '|DUPLICATE_LSOA_DOMAIN_MEASURE'
        WHEN BASE_REJECTION_REASON <> '' THEN BASE_REJECTION_REASON
        WHEN MEASURE_RANK > 1 THEN 'DUPLICATE_LSOA_DOMAIN_MEASURE'
    END AS REJECTION_REASON
FROM ASSESSED
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {WALES_LONG_CLEAN} AS
SELECT
    LSOA_CODE, LSOA_NAME, DOMAIN, MEASURE,
    MEASURE_VALUE::INTEGER AS MEASURE_VALUE,
    NOTES, 'WIMD' AS INDEX_NAME, 2025 AS INDEX_YEAR, 'Wales' AS NATION,
    SOURCE_FILE_NAME, '{RUN_ID}' AS CLEANING_RUN_ID,
    CURRENT_TIMESTAMP() AS CLEANED_AT
FROM STG_WALES_WIMD
WHERE REJECTION_REASON IS NULL
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {WALES_QUARANTINE} AS
SELECT *, '{RUN_ID}' AS CLEANING_RUN_ID, CURRENT_TIMESTAMP() AS QUARANTINED_AT
FROM STG_WALES_WIMD
WHERE REJECTION_REASON IS NOT NULL
""").collect()

domains = {
    "WIMD": "WIMD",
    "Income": "INCOME",
    "Employment": "EMPLOYMENT",
    "Education": "EDUCATION",
    "Health": "HEALTH",
    "Access to services": "ACCESS_TO_SERVICES",
    "Housing": "HOUSING",
    "Physical environment": "PHYSICAL_ENVIRONMENT",
    "Community safety": "COMMUNITY_SAFETY",
}
pivot_columns = []
for source_domain, output_prefix in domains.items():
    safe_domain = source_domain.replace("'", "''")
    for measure in ["Rank", "Decile"]:
        pivot_columns.append(
            f"MAX(IFF(DOMAIN = '{safe_domain}' AND MEASURE = '{measure}', "
            f"MEASURE_VALUE, NULL))::{ 'INTEGER' } AS {output_prefix}_{measure.upper()}"
        )

session.sql(f"""
CREATE OR REPLACE TABLE {WALES_CLEAN} AS
SELECT
    LSOA_CODE,
    MAX(LSOA_NAME) AS LSOA_NAME,
    {", ".join(pivot_columns)},
    TRUE AS EXCLUDE_COMMUNITY_SAFETY_FROM_CRIME_ANALYSIS,
    'WIMD' AS INDEX_NAME,
    2025 AS INDEX_YEAR,
    'Wales' AS NATION,
    'Confirm source LSOA boundary version before joining' AS GEOGRAPHY_VERSION,
    '{RUN_ID}' AS CLEANING_RUN_ID,
    CURRENT_TIMESTAMP() AS CLEANED_AT
FROM {WALES_LONG_CLEAN}
GROUP BY LSOA_CODE
""").collect()

## 5. Validation and audit

In [ ]:
checks = []

def add_check(dataset, name, severity, observed, expected, passed):
    record = {
        "dataset": dataset, "name": name, "severity": severity,
        "observed": str(observed), "expected": str(expected), "passed": bool(passed),
    }
    checks.append(record)
    observed_sql = record["observed"].replace("'", "''")
    expected_sql = record["expected"].replace("'", "''")
    session.sql(f"""
    INSERT INTO {VALIDATION_TABLE}
    SELECT '{RUN_ID}', '{dataset}', '{name}', '{severity}',
           '{observed_sql}', '{expected_sql}', {str(record["passed"]).upper()},
           CURRENT_TIMESTAMP()
    """).collect()

def scalar(sql, column="N"):
    return session.sql(sql).collect()[0][column]

# Population validations
pop_clean_rows = session.table(POPULATION_CLEAN).count()
pop_quarantine_rows = session.table(POPULATION_QUARANTINE).count()
pop_raw_rows = session.table("STG_POPULATION").count()
add_check("population", "row_reconciliation", "CRITICAL",
          pop_clean_rows + pop_quarantine_rows, pop_raw_rows,
          pop_clean_rows + pop_quarantine_rows == pop_raw_rows)
add_check("population", "expected_16_rows", "CRITICAL",
          pop_clean_rows, 16, pop_clean_rows == 16)
pop_duplicates = scalar(f"""
SELECT COUNT(*) AS N FROM (
    SELECT FORCE_NAME, YEAR FROM {POPULATION_CLEAN}
    GROUP BY FORCE_NAME, YEAR HAVING COUNT(*) > 1
)""")
add_check("population", "unique_force_year", "CRITICAL",
          pop_duplicates, 0, pop_duplicates == 0)

# England validations
eng_clean_rows = session.table(ENGLAND_CLEAN).count()
eng_quarantine_rows = session.table(ENGLAND_QUARANTINE).count()
eng_raw_rows = session.table("STG_ENGLAND_IMD").count()
add_check("england_imd", "row_reconciliation", "CRITICAL",
          eng_clean_rows + eng_quarantine_rows, eng_raw_rows,
          eng_clean_rows + eng_quarantine_rows == eng_raw_rows)
eng_duplicates = scalar(f"""
SELECT COUNT(*) AS N FROM (
    SELECT LSOA_CODE FROM {ENGLAND_CLEAN}
    GROUP BY LSOA_CODE HAVING COUNT(*) > 1
)""")
add_check("england_imd", "unique_lsoa", "CRITICAL",
          eng_duplicates, 0, eng_duplicates == 0)
eng_invalid_deciles = scalar(f"""
SELECT COUNT(*) AS N FROM {ENGLAND_CLEAN}
WHERE IMD_DECILE NOT BETWEEN 1 AND 10
   OR INCOME_DECILE NOT BETWEEN 1 AND 10
   OR EMPLOYMENT_DECILE NOT BETWEEN 1 AND 10
   OR EDUCATION_DECILE NOT BETWEEN 1 AND 10
   OR HEALTH_DECILE NOT BETWEEN 1 AND 10
   OR HOUSING_BARRIERS_DECILE NOT BETWEEN 1 AND 10
   OR LIVING_ENVIRONMENT_DECILE NOT BETWEEN 1 AND 10
""")
add_check("england_imd", "decile_ranges", "CRITICAL",
          eng_invalid_deciles, 0, eng_invalid_deciles == 0)

# Wales validations
wales_long_rows = session.table(WALES_LONG_CLEAN).count()
wales_quarantine_rows = session.table(WALES_QUARANTINE).count()
wales_raw_rows = session.table("STG_WALES_WIMD").count()
add_check("wales_wimd", "row_reconciliation", "CRITICAL",
          wales_long_rows + wales_quarantine_rows, wales_raw_rows,
          wales_long_rows + wales_quarantine_rows == wales_raw_rows)
add_check("wales_wimd", "expected_86265_long_rows", "CRITICAL",
          wales_long_rows, 86265, wales_long_rows == 86265)
wales_clean_rows = session.table(WALES_CLEAN).count()
add_check("wales_wimd", "expected_1917_lsoas", "CRITICAL",
          wales_clean_rows, 1917, wales_clean_rows == 1917)
wales_incomplete = scalar(f"""
SELECT COUNT(*) AS N FROM {WALES_CLEAN}
WHERE WIMD_RANK IS NULL OR WIMD_DECILE IS NULL
   OR INCOME_RANK IS NULL OR INCOME_DECILE IS NULL
   OR EMPLOYMENT_RANK IS NULL OR EMPLOYMENT_DECILE IS NULL
   OR EDUCATION_RANK IS NULL OR EDUCATION_DECILE IS NULL
   OR HEALTH_RANK IS NULL OR HEALTH_DECILE IS NULL
""")
add_check("wales_wimd", "core_pivot_completeness", "CRITICAL",
          wales_incomplete, 0, wales_incomplete == 0)

for dataset, raw_rows, clean_rows, quarantine_rows in [
    ("population", pop_raw_rows, pop_clean_rows, pop_quarantine_rows),
    ("england_imd", eng_raw_rows, eng_clean_rows, eng_quarantine_rows),
    ("wales_wimd", wales_raw_rows, wales_clean_rows, wales_quarantine_rows),
]:
    failed = [c for c in checks if c["dataset"] == dataset
              and c["severity"] == "CRITICAL" and not c["passed"]]
    status = "VALIDATED" if not failed else "VALIDATION_FAILED"
    session.sql(f"""
    INSERT INTO {AUDIT_TABLE}
    SELECT '{RUN_ID}', '{dataset}', TO_TIMESTAMP_TZ('{RUN_STARTED_AT.isoformat()}'),
           CURRENT_TIMESTAMP(), {raw_rows}, {clean_rows}, {quarantine_rows}, '{status}'
    """).collect()

session.sql(f"""
SELECT DATASET, CHECK_NAME, SEVERITY, OBSERVED_VALUE, EXPECTED_VALUE, PASSED
FROM {VALIDATION_TABLE}
WHERE RUN_ID = '{RUN_ID}'
ORDER BY DATASET, CHECK_NAME
""").show()

critical_failures = [c for c in checks if c["severity"] == "CRITICAL" and not c["passed"]]
if critical_failures:
    raise AssertionError(
        "Critical validation failure(s): "
        + ", ".join(f'{c["dataset"]}.{c["name"]}' for c in critical_failures)
    )
print("PASS - all supplementary datasets validated")

## 6. Export validated handoff files

Exports occur only after all critical checks pass. All supplementary files are written
to the existing clean stage used by the police-data pipeline.

In [ ]:
if EXPORT_TO_S3:
    exports = [
        (POPULATION_CLEAN, "population_clean.csv"),
        (ENGLAND_CLEAN, "deprivation_england_clean.csv"),
        (WALES_CLEAN, "deprivation_wales_clean.csv"),
    ]
    for table_name, file_name in exports:
        session.sql(f"""
        COPY INTO @{CLEAN_EXPORT_STAGE}/{file_name}
        FROM {table_name}
        FILE_FORMAT = (
            TYPE = CSV
            FIELD_OPTIONALLY_ENCLOSED_BY = '"'
            COMPRESSION = NONE
            NULL_IF = ('')
        )
        HEADER = TRUE
        SINGLE = TRUE
        OVERWRITE = TRUE
        """).collect()
        print(f"Exported @{CLEAN_EXPORT_STAGE}/{file_name}")

    session.sql(f"""
    COPY INTO @{CLEAN_EXPORT_STAGE}/supplementary_cleaning_audit.csv
    FROM (
        SELECT *
        FROM {AUDIT_TABLE}
        WHERE RUN_ID = '{RUN_ID}'
        ORDER BY DATASET
    )
    FILE_FORMAT = (
        TYPE = CSV
        FIELD_OPTIONALLY_ENCLOSED_BY = '"'
        COMPRESSION = NONE
        NULL_IF = ('')
    )
    HEADER = TRUE
    SINGLE = TRUE
    OVERWRITE = TRUE
    """).collect()

    session.sql(f"LIST @{CLEAN_EXPORT_STAGE}").show()
else:
    print("S3 export disabled. Validated Snowflake tables were still published.")

## Handoff notes

- Population is observed only for 2021-2024; `IS_ESTIMATED` is always false.
- England IMD uses 2021 LSOA geography.
- Confirm the Wales LSOA boundary version before joining.
- Never compare raw England IMD scores directly with Wales WIMD ranks.
- Exclude England crime-domain and Wales community-safety measures when analysing
  police crime to avoid circularity.
- Population must join on `FORCE_NAME + YEAR`.
- Deprivation must join on `LSOA_CODE`.
- Before and after each left join, reconcile crime row counts and report match rates.